In [ ]:
# SPR 2026 - BERTimbau Large + AWP + Focal Loss + Optuna (modelo proprio)
# AWP (Adversarial Weight Perturbation) perturba os pesos do modelo
# durante o treino, forcando o modelo a ser robusto a pequenas mudancas.
# Tipicamente adiciona 0.3-0.5 pontos em classificacao de texto.

import os, re, gc, pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, AutoConfig,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from tqdm.auto import tqdm

# CONFIG
SEED        = 42
MAX_LEN     = 512
BATCH_SIZE  = 8
EPOCHS      = 5
LR          = 2e-5
WEIGHT_DECAY= 0.01
WARMUP_RATIO= 0.1
N_FOLDS     = 5
NUM_CLASSES = 7
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
N_TRIALS    = 300
LABEL_SMOOTHING = 0.05

# AWP config
AWP_LR = 1e-1
AWP_EPS = 1e-2
AWP_START_EPOCH = 1  # start AWP from epoch 2 (let model warm up first)

DATA_DIR = Path('/kaggle/input/competitions/spr-2026-mammography-report-classification')
MODEL_CANDIDATES = [
    '/kaggle/input/models/fabianofilho/bertimbau-ptbr-complete/pytorch/default/1',
    '/kaggle/input/bertimbau-ptbr-complete/pytorch/bert-large-portuguese-cased/1',
    '/kaggle/input/bert-large-portuguese-cased',
]
MODEL_PATH = next((p for p in MODEL_CANDIDATES if Path(p).exists()), None)
assert MODEL_PATH, 'BERTimbau Large nao encontrado. Adicione como Input.'

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_FP16 = torch.cuda.is_available()
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device: {DEVICE}, FP16: {USE_FP16}')
print(f'Model: {MODEL_PATH}')
print(f'AWP: lr={AWP_LR}, eps={AWP_EPS}, start_epoch={AWP_START_EPOCH}')

In [ ]:
# ========== Data Loading + Preprocessing ==========

INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicacao:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']


def clean_text(text):
    """Limpa texto do laudo: normaliza espacos, remove lixo."""
    if not isinstance(text, str):
        return ''
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\sà-ü.,;:!?()\[\]/\\\-]', '', text)
    return text.strip()


def extract_section(text, markers, next_markers=None):
    """Extrai secao do laudo entre markers."""
    text_lower = text.lower()
    start = -1
    for m in markers:
        idx = text_lower.find(m)
        if idx != -1:
            start = idx + len(m)
            break
    if start == -1:
        return ''
    end = len(text)
    if next_markers:
        for m in next_markers:
            idx = text_lower.find(m, start)
            if idx != -1 and idx < end:
                end = idx
    return text[start:end].strip()


def build_input_text(row):
    """Constroi texto de entrada concatenando secoes do laudo."""
    text = str(row.get('texto', '') or row.get('text', '') or '')
    cleaned = clean_text(text)

    indicacao = extract_section(cleaned, INDICACAO_MARKERS, ACHADOS_MARKERS + COMPARATIVA_MARKERS)
    achados = extract_section(cleaned, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(cleaned, COMPARATIVA_MARKERS)

    parts = []
    if indicacao:
        parts.append(f'[INDICACAO] {indicacao}')
    if achados:
        parts.append(f'[ACHADOS] {achados}')
    if comparativa:
        parts.append(f'[COMPARATIVA] {comparativa}')

    if parts:
        return ' '.join(parts)
    return cleaned


# Load data
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')

# Detect text column
text_col = 'texto' if 'texto' in train_df.columns else 'text'
label_col = 'label' if 'label' in train_df.columns else 'birads'

print(f'Text column: {text_col}, Label column: {label_col}')
print(f'Label distribution:\n{train_df[label_col].value_counts().sort_index()}')

# Build input texts
train_df['input_text'] = train_df.apply(build_input_text, axis=1)
test_df['input_text']  = test_df.apply(build_input_text, axis=1)

labels = train_df[label_col].values
print(f'\nSample input text (first 200 chars):\n{train_df["input_text"].iloc[0][:200]}')

In [ ]:
# ========== Dataset + FocalLoss with Label Smoothing ==========

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)


class MammographyDataset(Dataset):
    """Dataset para classificacao de laudos de mamografia."""

    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].squeeze(0)
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


class FocalLossWithSmoothing(nn.Module):
    """Focal Loss com label smoothing para classes desbalanceadas."""

    def __init__(self, alpha=0.25, gamma=2.0, smoothing=0.05, num_classes=7):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
        self.num_classes = num_classes

    def forward(self, logits, targets):
        # Label smoothing
        with torch.no_grad():
            smooth_targets = torch.zeros_like(logits)
            smooth_targets.fill_(self.smoothing / (self.num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)

        log_probs = F.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)

        # Focal modulation
        focal_weight = (1.0 - probs) ** self.gamma

        # Weighted loss
        loss = -self.alpha * focal_weight * smooth_targets * log_probs
        return loss.sum(dim=1).mean()


print(f'Tokenizer loaded: {tokenizer.__class__.__name__}')
print(f'Vocab size: {tokenizer.vocab_size}')

In [ ]:
# ========== AWP (Adversarial Weight Perturbation) ==========


class AWP:
    """Adversarial Weight Perturbation.

    Perturba os pesos do modelo na direcao do gradiente durante o treino,
    forcando o modelo a encontrar minimos mais planos no loss landscape.
    Isso melhora a generalizacao, tipicamente +0.3-0.5 pontos em F1.
    """

    def __init__(self, model, optimizer, adv_lr=1e-1, adv_eps=1e-2):
        self.model = model
        self.optimizer = optimizer
        self.adv_lr = adv_lr
        self.adv_eps = adv_eps
        self.backup = {}
        self.backup_eps = {}

    def attack_backward(self, inputs, labels, criterion):
        """Executa o passo adversarial completo.

        1. Salva pesos originais
        2. Perturba pesos na direcao do gradiente
        3. Forward no modelo perturbado
        4. Backward no loss perturbado (acumula gradiente)
        5. Restaura pesos originais
        """
        # 1. Save current weights
        self._save()
        # 2. Perturb weights
        self._attack_step()
        # 3. Forward on perturbed model
        with torch.amp.autocast('cuda', enabled=USE_FP16):
            outputs = self.model(**inputs)
            adv_loss = criterion(outputs.logits, labels)
        # 4. Backward on perturbed loss
        scaler.scale(adv_loss).backward()
        # 5. Restore original weights
        self._restore()

    def _attack_step(self):
        e = 1e-6
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None:
                norm1 = torch.norm(param.grad)
                norm2 = torch.norm(param.data.detach())
                if norm1 != 0 and not torch.isnan(norm1):
                    r_at = self.adv_lr * param.grad / (norm1 + e) * (norm2 + e)
                    param.data.add_(r_at)
                    param.data = torch.min(
                        torch.max(param.data, self.backup_eps[name][0]),
                        self.backup_eps[name][1]
                    )

    def _save(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None:
                if name not in self.backup:
                    self.backup[name] = param.data.clone()
                    grad_eps = self.adv_eps * param.data.abs().detach()
                    self.backup_eps[name] = (
                        self.backup[name] - grad_eps,
                        self.backup[name] + grad_eps,
                    )

    def _restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}
        self.backup_eps = {}


print('AWP class ready.')

In [ ]:
# ========== Training Helpers (with AWP integration) ==========

scaler = torch.amp.GradScaler('cuda', enabled=USE_FP16)


def train_epoch(model, loader, optimizer, scheduler, criterion, awp=None, epoch=0):
    """Treina uma epoca com FP16 + AWP adversarial.

    Se epoch >= AWP_START_EPOCH e awp nao for None, executa o passo adversarial
    apos o backward normal. O gradiente adversarial se acumula com o gradiente
    normal antes do optimizer.step().
    """
    model.train()
    total_loss = 0.0
    total_steps = 0
    use_awp = (awp is not None) and (epoch >= AWP_START_EPOCH)

    for batch in tqdm(loader, desc=f'  Train (AWP={use_awp})', leave=False):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels_batch = batch['labels'].to(DEVICE)

        kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)

        optimizer.zero_grad()

        # Normal forward + backward
        with torch.amp.autocast('cuda', enabled=USE_FP16):
            outputs = model(**kwargs)
            loss = criterion(outputs.logits, labels_batch)

        scaler.scale(loss).backward()

        # AWP adversarial step (after normal backward, before optimizer step)
        if use_awp:
            awp.attack_backward(kwargs, labels_batch, criterion)

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        total_steps += 1

    return total_loss / max(total_steps, 1)


@torch.no_grad()
def evaluate(model, loader):
    """Avalia modelo, retorna probs e labels."""
    model.eval()
    all_probs = []
    all_labels = []

    for batch in tqdm(loader, desc='  Eval', leave=False):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)

        kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)

        with torch.amp.autocast('cuda', enabled=USE_FP16):
            outputs = model(**kwargs)

        probs = F.softmax(outputs.logits, dim=1).cpu().numpy()
        all_probs.append(probs)

        if 'labels' in batch:
            all_labels.append(batch['labels'].numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    if all_labels:
        all_labels = np.concatenate(all_labels, axis=0)
    else:
        all_labels = None
    return all_probs, all_labels


def compute_f1(probs, labels):
    """Calcula macro F1 a partir de probs e labels."""
    preds = np.argmax(probs, axis=1)
    return f1_score(labels, preds, average='macro')


print('Training helpers ready (with AWP).')

In [ ]:
# ========== Calibration Functions ==========

def temperature_scale(logits, temperature):
    """Aplica temperature scaling nos logits."""
    return logits / temperature


def apply_thresholds(probs, thresholds):
    """Aplica multiplicadores de threshold por classe e retorna preds."""
    adjusted = probs * np.array(thresholds)
    return np.argmax(adjusted, axis=1)


def optimize_calibration(oof_probs, oof_labels, n_trials=300):
    """Otimiza temperatura + thresholds com Optuna.

    8 dimensoes: 1 temperatura + 7 multiplicadores de threshold.
    """

    # Converter probs para logits (log space)
    oof_logits = np.log(np.clip(oof_probs, 1e-7, 1.0))

    def objective(trial):
        temp = trial.suggest_float('temperature', 0.3, 2.5)
        thresholds = []
        for c in range(NUM_CLASSES):
            t = trial.suggest_float(f'th_{c}', 0.05, 3.0)
            thresholds.append(t)

        scaled_logits = temperature_scale(oof_logits, temp)
        scaled_probs = np.exp(scaled_logits) / np.exp(scaled_logits).sum(axis=1, keepdims=True)
        preds = apply_thresholds(scaled_probs, thresholds)
        return f1_score(oof_labels, preds, average='macro')

    study = optuna.create_study(direction='maximize', seed=SEED)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best = study.best_params
    best_temp = best['temperature']
    best_thresholds = [best[f'th_{c}'] for c in range(NUM_CLASSES)]
    best_f1 = study.best_value

    return best_temp, best_thresholds, best_f1


print('Calibration functions ready.')

In [ ]:
# ========== 5-Fold Training Loop with AWP ==========

train_texts = train_df['input_text'].values
test_texts  = test_df['input_text'].values

oof_probs  = np.zeros((len(train_df), NUM_CLASSES))
test_probs = np.zeros((len(test_df), NUM_CLASSES))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

criterion = FocalLossWithSmoothing(
    alpha=FOCAL_ALPHA,
    gamma=FOCAL_GAMMA,
    smoothing=LABEL_SMOOTHING,
    num_classes=NUM_CLASSES,
)

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_texts, labels)):
    print(f'\n{"="*60}')
    print(f'Fold {fold + 1}/{N_FOLDS}')
    print(f'{"="*60}')

    # Datasets
    train_dataset = MammographyDataset(train_texts[train_idx], labels[train_idx])
    val_dataset   = MammographyDataset(train_texts[val_idx], labels[val_idx])
    test_dataset  = MammographyDataset(test_texts)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)

    # Fresh model from pretrained
    config = AutoConfig.from_pretrained(MODEL_PATH, num_labels=NUM_CLASSES, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH,
        config=config,
        ignore_mismatched_sizes=True,
        local_files_only=True,
    ).to(DEVICE)

    # Optimizer + Scheduler
    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
    optimizer_grouped = [
        {
            'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            'weight_decay': WEIGHT_DECAY,
        },
        {
            'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0,
        },
    ]
    optimizer = torch.optim.AdamW(optimizer_grouped, lr=LR)

    total_steps = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    # AWP instance for this fold
    awp = AWP(model, optimizer, adv_lr=AWP_LR, adv_eps=AWP_EPS)

    # Training
    best_val_f1 = 0.0
    best_state = None

    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, awp=awp, epoch=epoch)
        val_probs, val_labels = evaluate(model, val_loader)
        val_f1 = compute_f1(val_probs, val_labels)

        awp_status = 'ON' if epoch >= AWP_START_EPOCH else 'OFF'
        print(f'  Epoch {epoch + 1}/{EPOCHS}: loss={train_loss:.4f}, val_f1={val_f1:.4f}, AWP={awp_status}')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f'  Best val F1: {best_val_f1:.4f}')
    fold_scores.append(best_val_f1)

    # Load best and get OOF + test predictions
    model.load_state_dict(best_state)
    model.to(DEVICE)

    val_probs_best, _ = evaluate(model, val_loader)
    oof_probs[val_idx] = val_probs_best

    fold_test_probs, _ = evaluate(model, test_loader)
    test_probs += fold_test_probs / N_FOLDS

    # Cleanup
    del model, optimizer, scheduler, best_state, awp
    gc.collect()
    torch.cuda.empty_cache()

# Overall OOF F1
oof_f1_raw = compute_f1(oof_probs, labels)
print(f'\n{"="*60}')
print(f'Overall OOF F1 (raw argmax): {oof_f1_raw:.4f}')
print(f'Fold scores: {[f"{s:.4f}" for s in fold_scores]}')
print(f'Mean fold F1: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})')

In [ ]:
# ========== Optuna Calibration + Submission ==========

print('Running Optuna calibration (300 trials)...')
best_temp, best_thresholds, best_f1_calibrated = optimize_calibration(
    oof_probs, labels, n_trials=N_TRIALS
)

print(f'\nCalibration results:')
print(f'  Temperature: {best_temp:.4f}')
print(f'  Thresholds: {[f"{t:.4f}" for t in best_thresholds]}')
print(f'\nComparison:')
print(f'  Raw argmax F1:   {oof_f1_raw:.4f}')
print(f'  Calibrated F1:   {best_f1_calibrated:.4f}')
print(f'  Improvement:     {best_f1_calibrated - oof_f1_raw:+.4f}')

# Save calibration params
calib_params = {
    'temperature': best_temp,
    'thresholds': best_thresholds,
    'oof_f1_raw': oof_f1_raw,
    'oof_f1_calibrated': best_f1_calibrated,
}
with open('calibration_params.pkl', 'wb') as f:
    pickle.dump(calib_params, f)
print('\nCalibration params saved to calibration_params.pkl')

# Apply calibrated temperature + thresholds to test probs
test_logits = np.log(np.clip(test_probs, 1e-7, 1.0))
scaled_logits = temperature_scale(test_logits, best_temp)
scaled_probs = np.exp(scaled_logits) / np.exp(scaled_logits).sum(axis=1, keepdims=True)
test_preds = apply_thresholds(scaled_probs, best_thresholds)

# Build submission
id_col = 'id' if 'id' in test_df.columns else test_df.columns[0]
sub = pd.DataFrame({
    id_col: test_df[id_col],
    label_col: test_preds,
})

sub.to_csv('submission.csv', index=False)
print(f'\nSubmission shape: {sub.shape}')
print(f'Prediction distribution:\n{sub[label_col].value_counts().sort_index()}')
print(f'\nSaved to submission.csv')
print(sub.head(10))